# HTD OOF M1 Held-Out Prediction

Kaggle inference notebook for one OOF crop-LoRA model. Set `FOLD_ID` to the model you trained, mount the matching LoRA output and `oof_stage2b_crops`, then run all cells. The notebook predicts the held-out part only and writes `submission.csv` with the same columns as the competition submission: `image,regions`.

Fold mapping:

- `FOLD_ID = 1`: trained on parts 1+2, predicts part 3.
- `FOLD_ID = 2`: trained on parts 1+3, predicts part 2.
- `FOLD_ID = 3`: trained on parts 2+3, predicts part 1.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '--upgrade-strategy',
            'only-if-needed',
            'accelerate',
            'peft',
            'bitsandbytes',
            'qwen-vl-utils',
            'pandas==2.2.2',
            'pillow<12',
        ],
        [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'git+https://github.com/huggingface/transformers.git'],
    ]
    for cmd in commands:
        print('Running:', ' '.join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import torch

os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# EDIT THIS BLOCK ON KAGGLE
# =========================
# FOLD_ID=1 trained parts 1+2 and predicts part 3.
# FOLD_ID=2 trained parts 1+3 and predicts part 2.
# FOLD_ID=3 trained parts 2+3 and predicts part 1.
FOLD_ID = 1

# Optional explicit paths. Leave blank to auto-discover under /kaggle/input and /kaggle/working.
BASE_MODEL_PATH = ''
LORA_DIR = ''
CROP_DATA_ROOT = ''
METADATA_RECORDS_PATH = ''
CHECKPOINT_INPUT_DIR = ''

OUTPUT_CSV = 'submission.csv'
INSTALL_NOTEBOOK_OUTPUT_NAME = True

# T4x2: each process loads one 4-bit Qwen+LoRA copy on its GPU.
MAX_GPUS = 2
BATCH_SIZE = 2
MAX_NEW_TOKENS = 192
MAX_PIXELS_CROP = 320_000
CHECKPOINT_EVERY = 100
PROGRESS_LOG_EVERY = 25
RESUME_PARTIALS = True

TEST_MODE = False
TEST_LIMIT = 64
USE_METADATA_TEXT_FOR_UNPREDICTED = False
# =========================
# END EDIT BLOCK
# =========================

FOLD_TO_TRAIN_PARTS = {
    1: [1, 2],
    2: [1, 3],
    3: [2, 3],
}
if FOLD_ID not in FOLD_TO_TRAIN_PARTS:
    raise ValueError(f'FOLD_ID must be one of {sorted(FOLD_TO_TRAIN_PARTS)}, got {FOLD_ID}')

TRAIN_PARTS = FOLD_TO_TRAIN_PARTS[FOLD_ID]
HELD_OUT_PART = sorted(set([1, 2, 3]) - set(TRAIN_PARTS))[0]
RUN_NAME = f'model{FOLD_ID}_parts{"_".join(str(p) for p in TRAIN_PARTS)}'
PARTIAL_PREFIX = f'oof_{RUN_NAME}_heldout_part{HELD_OUT_PART}_partial_gpu'
PREDICTION_CSV = f'oof_{RUN_NAME}_heldout_part{HELD_OUT_PART}_crop_predictions.csv'

BASE_MODEL_CANDIDATES = [
    BASE_MODEL_PATH,
    '/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1',
    '/kaggle/input/qwen3-vl-8b-instruct',
    'Qwen/Qwen3-VL-8B-Instruct',
]

LORA_CANDIDATES = [
    LORA_DIR,
    f'/kaggle/input/htd-oof-{RUN_NAME}/qwen3vl_rukopys_oof_{RUN_NAME}_lora_final',
    f'/kaggle/input/htd-oof-{RUN_NAME}/{RUN_NAME}/qwen3vl_rukopys_oof_{RUN_NAME}_lora_final',
    f'/kaggle/input/htd-oof-m1/{RUN_NAME}/qwen3vl_rukopys_oof_{RUN_NAME}_lora_final',
    f'/kaggle/input/htd-oof-m1/qwen3vl_rukopys_oof_{RUN_NAME}_lora_final',
]

CROP_ROOT_CANDIDATES = [
    CROP_DATA_ROOT,
    '/kaggle/working/oof_stage2b_crops',
    '/kaggle/input/oof_stage2b_crops',
]

VALID_TYPES = {'handwritten', 'printed', 'formula', 'table', 'annotation', 'image', 'graph'}
SCORABLE_TYPES = {'handwritten', 'printed', 'formula', 'table', 'annotation'}
IMAGE_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.webp', '.bmp']

warnings.filterwarnings('ignore', message=r'.*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*')

print('FOLD_ID:', FOLD_ID)
print('Train parts:', TRAIN_PARTS)
print('Held-out part:', HELD_OUT_PART)
print('Run name:', RUN_NAME)
print('Output CSV:', OUTPUT_CSV)


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_type(value):
    value = str(value or 'handwritten').strip().lower()
    return value if value in VALID_TYPES else 'handwritten'


def canonical_file_name(value):
    return str(value or '').replace('\\', '/')


def canonical_bbox(box):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return [0, 0, 0, 0]
    out = []
    for value in box:
        try:
            out.append(int(round(float(value))))
        except Exception:
            out.append(0)
    return out


def sample_key_from_parts(file_name, bbox, region_type):
    return json.dumps(
        {
            'file': canonical_file_name(file_name),
            'bbox': canonical_bbox(bbox),
            'type': normalize_type(region_type),
        },
        sort_keys=True,
        ensure_ascii=False,
    )


def sample_key(row):
    file_name = row.get('source_file_name') or row.get('file_name') or row.get('source_image_path') or ''
    return sample_key_from_parts(file_name, row.get('bbox'), row.get('region_type') or row.get('type'))


def part_dir(root, part):
    return Path(root) / f'oof_part{part}'


def part_manifest(root, part):
    return part_dir(root, part) / 'stage2b_oof_crop_samples.jsonl'


def part_records(root, part):
    return part_dir(root, part) / 'metadata_records.jsonl'


def is_lora_dir(path):
    path = Path(path)
    if not path.exists():
        return False
    has_config = (path / 'adapter_config.json').exists()
    has_weights = (path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists()
    return has_config and has_weights


def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        item = str(item or '').strip()
        if not item:
            continue
        if item.startswith('/') and Path(item).exists():
            return item
        if not item.startswith('/'):
            return item
    raise FileNotFoundError('No Qwen3-VL base model found. Set BASE_MODEL_PATH or add Qwen3-VL as Kaggle input.')


def lora_score(path):
    lower = str(path).lower()
    score = 0
    if RUN_NAME.lower() in lower:
        score += 20
    if f'model{FOLD_ID}' in lower:
        score += 5
    if 'lora_final' in lower or 'final' in lower:
        score += 3
    if 'checkpoint' in lower:
        score -= 5
    return score


def find_lora_dir():
    for item in LORA_CANDIDATES:
        item = str(item or '').strip()
        if item and is_lora_dir(item):
            return Path(item)

    found = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not root.exists():
            continue
        for cfg in root.rglob('adapter_config.json'):
            candidate = cfg.parent
            if is_lora_dir(candidate):
                found.append(candidate)
    if found:
        found = sorted(found, key=lambda p: (lora_score(p), -len(str(p))), reverse=True)
        print('Auto-discovered LoRA candidates:')
        for p in found[:8]:
            print(' ', p, 'score=', lora_score(p))
        return found[0]
    raise FileNotFoundError('No LoRA adapter found. Mount the trained M1 LoRA output as a Kaggle input or set LORA_DIR.')


def root_has_part(root, part):
    return part_manifest(root, part).exists()


def discover_crop_data_root(part):
    for item in CROP_ROOT_CANDIDATES:
        item = str(item or '').strip()
        if item and root_has_part(Path(item), part):
            return Path(item)

    scan_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    seen = set()
    for scan_root in scan_roots:
        if not scan_root.exists():
            continue
        for manifest in scan_root.rglob('stage2b_oof_crop_samples.jsonl'):
            if manifest.parent.name != f'oof_part{part}':
                continue
            root = manifest.parent.parent
            key = str(root.resolve()) if root.exists() else str(root)
            if key in seen:
                continue
            seen.add(key)
            if root_has_part(root, part):
                return root
    raise FileNotFoundError(f'Could not find oof_part{part}/stage2b_oof_crop_samples.jsonl. Mount oof_stage2b_crops or set CROP_DATA_ROOT.')


def discover_metadata_records(root, part):
    explicit = str(METADATA_RECORDS_PATH or '').strip()
    if explicit:
        path = Path(explicit)
        if path.exists():
            return path
        raise FileNotFoundError(f'METADATA_RECORDS_PATH does not exist: {path}')

    in_part = part_records(root, part)
    if in_part.exists():
        return in_part

    wanted = f'metadata_part{part}.jsonl'
    for scan_root in [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]:
        if not scan_root.exists():
            continue
        matches = sorted(scan_root.rglob(wanted))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'Could not find metadata_records.jsonl or {wanted}.')


def resolve_cached_image_path(root, part, row):
    part_root = part_dir(root, part)
    raw = Path(row.get('image_path') or row.get('relative_image_path') or '')
    if raw.is_absolute() and raw.exists():
        return str(raw)

    candidates = [part_root / raw, Path(root) / raw]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError(f'Cached crop not found for part {part}: {raw} under {part_root}')


def load_part_samples(root, part):
    rows = read_jsonl(part_manifest(root, part))
    out = []
    for idx, row in enumerate(rows):
        row = dict(row)
        row['image_path'] = resolve_cached_image_path(root, part, row)
        row['fold_part'] = part
        row['_sample_key'] = sample_key(row)
        row['_manifest_index'] = idx
        out.append(row)
    return out


model_id = find_model_id()
lora_dir = find_lora_dir()
crop_data_root = discover_crop_data_root(HELD_OUT_PART)
metadata_path = discover_metadata_records(crop_data_root, HELD_OUT_PART)
heldout_samples = load_part_samples(crop_data_root, HELD_OUT_PART)
metadata_records = read_jsonl(metadata_path)

if TEST_MODE:
    heldout_samples = heldout_samples[:TEST_LIMIT]
    keep_files = {canonical_file_name(s.get('source_file_name')) for s in heldout_samples}
    metadata_records = [r for r in metadata_records if canonical_file_name(r.get('file_name')) in keep_files]

if not heldout_samples:
    raise RuntimeError('No held-out crop samples loaded.')
if not metadata_records:
    raise RuntimeError('No metadata records loaded.')

print('Base model:', model_id)
print('LoRA:', lora_dir)
print('Crop data root:', crop_data_root)
print('Metadata records:', metadata_path)
print('Held-out crop samples:', len(heldout_samples))
print('Held-out pages:', len(metadata_records))
print('Samples by type:', dict(sorted(Counter(normalize_type(s.get('region_type')) for s in heldout_samples).items())))
print('First sample:', {k: heldout_samples[0].get(k) for k in ['source_file_name', 'bbox', 'region_type', 'image_path']})


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = 'left'
    return processor


def load_worker_model(device):
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={'': device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation='sdpa',
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor


def apply_chat_template(processor, messages):
    candidates = [
        {'tokenize': False, 'add_generation_prompt': True, 'template_kwargs': {'enable_thinking': False}},
        {'tokenize': False, 'add_generation_prompt': True, 'processor_kwargs': {'enable_thinking': False}},
        {'tokenize': False, 'add_generation_prompt': True, 'enable_thinking': False},
        {'tokenize': False, 'add_generation_prompt': True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', message=r'.*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*')
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def build_messages(sample):
    return [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': sample['image_path'], 'max_pixels': MAX_PIXELS_CROP},
                {'type': 'text', 'text': sample['prompt']},
            ],
        }
    ]


def clean_prediction(text):
    text = str(text or '').replace('\r\n', '\n').replace('\r', '\n').strip()
    if text.startswith('```') and text.endswith('```'):
        lines = text.splitlines()
        if len(lines) >= 2:
            text = '\n'.join(lines[1:-1]).strip()
    return text

def generate_batch(model, processor, samples, device):
    messages_batch = [build_messages(sample) for sample in samples]
    processor.tokenizer.padding_side = 'left'
    texts = [apply_chat_template(processor, messages) for messages in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={'padding': True, 'return_tensors': 'pt'},
            images_kwargs={'return_tensors': 'pt'},
            videos_kwargs={'return_tensors': 'pt'},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors='pt',
        )
    inputs = inputs.to(device)
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return [clean_prediction(x) for x in decoded]


In [ ]:
def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def detect_num_gpus():
    try:
        out = subprocess.check_output(['nvidia-smi', '-L']).decode('utf-8').strip()
        count = len([line for line in out.splitlines() if line.strip()])
    except Exception:
        count = torch.cuda.device_count()
    return max(1, min(MAX_GPUS, count))


WORKER_SCRIPT = Path.cwd() / 'oof_heldout_crop_worker.py'
WORKER_CONFIG = Path.cwd() / f'{PARTIAL_PREFIX}_worker_config.json'

worker_source = 'import argparse\nimport gc\nimport json\nimport os\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom peft import PeftModel\nfrom qwen_vl_utils import process_vision_info\nfrom transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig\n\nos.environ[\'TRANSFORMERS_VERBOSITY\'] = \'error\'\nos.environ[\'TRANSFORMERS_NO_ADVISORY_WARNINGS\'] = \'1\'\nos.environ[\'PYTORCH_ALLOC_CONF\'] = \'expandable_segments:True\'\nwarnings.filterwarnings(\'ignore\', message=r\'.*Kwargs passed to `processor\\.__call__` have to be in `processor_kwargs` dict.*\')\n\nVALID_TYPES = {\'handwritten\', \'printed\', \'formula\', \'table\', \'annotation\', \'image\', \'graph\'}\n\n\ndef read_jsonl(path):\n    rows = []\n    with open(path, \'r\', encoding=\'utf-8\') as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                rows.append(json.loads(line))\n    return rows\n\n\ndef normalize_type(value):\n    value = str(value or \'handwritten\').strip().lower()\n    return value if value in VALID_TYPES else \'handwritten\'\n\n\ndef canonical_file_name(value):\n    return str(value or \'\').replace(\'\\\\\', \'/\')\n\n\ndef canonical_bbox(box):\n    if not isinstance(box, (list, tuple)) or len(box) != 4:\n        return [0, 0, 0, 0]\n    out = []\n    for value in box:\n        try:\n            out.append(int(round(float(value))))\n        except Exception:\n            out.append(0)\n    return out\n\n\ndef configure_processor_for_generation(processor):\n    if processor.tokenizer.pad_token_id is None:\n        processor.tokenizer.pad_token = processor.tokenizer.eos_token\n    processor.tokenizer.padding_side = \'left\'\n    return processor\n\n\ndef load_worker_model(model_id, lora_dir, device):\n    quantization_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\'nf4\',\n    )\n    base = AutoModelForImageTextToText.from_pretrained(\n        model_id,\n        device_map={\'\': device},\n        quantization_config=quantization_config,\n        dtype=torch.float16,\n        trust_remote_code=True,\n        attn_implementation=\'sdpa\',\n        low_cpu_mem_usage=True,\n    )\n    model = PeftModel.from_pretrained(base, str(lora_dir))\n    model.eval()\n    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)\n    processor = configure_processor_for_generation(processor)\n    if processor.tokenizer.pad_token_id is not None:\n        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id\n    return model, processor\n\n\ndef apply_chat_template(processor, messages):\n    candidates = [\n        {\'tokenize\': False, \'add_generation_prompt\': True, \'template_kwargs\': {\'enable_thinking\': False}},\n        {\'tokenize\': False, \'add_generation_prompt\': True, \'processor_kwargs\': {\'enable_thinking\': False}},\n        {\'tokenize\': False, \'add_generation_prompt\': True, \'enable_thinking\': False},\n        {\'tokenize\': False, \'add_generation_prompt\': True},\n    ]\n    for kwargs in candidates:\n        try:\n            with warnings.catch_warnings():\n                warnings.filterwarnings(\'ignore\', message=r\'.*Kwargs passed to `processor\\.__call__` have to be in `processor_kwargs` dict.*\')\n                return processor.apply_chat_template(messages, **kwargs)\n        except TypeError:\n            continue\n    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef build_messages(sample, max_pixels_crop):\n    return [\n        {\n            \'role\': \'user\',\n            \'content\': [\n                {\'type\': \'image\', \'image\': sample[\'image_path\'], \'max_pixels\': max_pixels_crop},\n                {\'type\': \'text\', \'text\': sample[\'prompt\']},\n            ],\n        }\n    ]\n\n\ndef clean_prediction(text):\n    text = str(text or \'\').replace(\'\\r\\n\', \'\\n\').replace(\'\\r\', \'\\n\').strip()\n    if text.startswith(\'```\') and text.endswith(\'```\'):\n        lines = text.splitlines()\n        if len(lines) >= 2:\n            text = \'\\n\'.join(lines[1:-1]).strip()\n    return text\n\n\ndef generate_batch(model, processor, samples, device, max_pixels_crop, max_new_tokens):\n    messages_batch = [build_messages(sample, max_pixels_crop) for sample in samples]\n    processor.tokenizer.padding_side = \'left\'\n    texts = [apply_chat_template(processor, messages) for messages in messages_batch]\n    image_inputs, video_inputs = process_vision_info(messages_batch)\n    try:\n        inputs = processor(\n            text=texts,\n            images=image_inputs,\n            videos=video_inputs,\n            text_kwargs={\'padding\': True, \'return_tensors\': \'pt\'},\n            images_kwargs={\'return_tensors\': \'pt\'},\n            videos_kwargs={\'return_tensors\': \'pt\'},\n        )\n    except TypeError:\n        inputs = processor(\n            text=texts,\n            images=image_inputs,\n            videos=video_inputs,\n            padding=True,\n            return_tensors=\'pt\',\n        )\n    inputs = inputs.to(device)\n    with torch.no_grad(), torch.amp.autocast(\'cuda\', dtype=torch.float16):\n        out = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            num_beams=1,\n        )\n    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]\n    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n    del inputs, out, trimmed\n    torch.cuda.empty_cache()\n    return [clean_prediction(x) for x in decoded]\n\n\ndef result_row(sample, text):\n    return {\n        \'sample_key\': sample[\'_sample_key\'],\n        \'source_file_name\': canonical_file_name(sample.get(\'source_file_name\')),\n        \'bbox\': json.dumps(canonical_bbox(sample.get(\'bbox\')), ensure_ascii=False),\n        \'region_type\': normalize_type(sample.get(\'region_type\')),\n        \'text\': clean_prediction(text),\n    }\n\n\ndef save_partial(path, rows):\n    frame = pd.DataFrame(rows)\n    if frame.empty:\n        frame = pd.DataFrame(columns=[\'sample_key\', \'source_file_name\', \'bbox\', \'region_type\', \'text\'])\n    frame = frame.drop_duplicates(subset=[\'sample_key\'], keep=\'last\')\n    tmp = str(path) + \'.tmp\'\n    frame.to_csv(tmp, index=False)\n    os.replace(tmp, path)\n\n\ndef load_partial(path, resume_partials):\n    if not resume_partials or not Path(path).exists():\n        return set(), []\n    try:\n        old = pd.read_csv(path).fillna(\'\')\n        old = old.drop_duplicates(subset=[\'sample_key\'], keep=\'last\')\n        rows = old.to_dict(\'records\')\n        done = set(old[\'sample_key\'].astype(str).tolist())\n        return done, rows\n    except Exception as exc:\n        print(f\'Could not read partial {path}: {exc}\', flush=True)\n        return set(), []\n\n\ndef format_duration(seconds):\n    if seconds is None or seconds <= 0:\n        return \'--:--\'\n    seconds = int(round(seconds))\n    hours = seconds // 3600\n    minutes = (seconds % 3600) // 60\n    secs = seconds % 60\n    if hours:\n        return f\'{hours:02d}:{minutes:02d}:{secs:02d}\'\n    return f\'{minutes:02d}:{secs:02d}\'\n\n\ndef print_progress(gpu_id, done, total, run_done, elapsed, last_name):\n    pct = 100.0 * done / max(1, total)\n    speed = run_done / elapsed if run_done > 0 and elapsed > 0 else 0.0\n    remaining = max(0, total - done)\n    eta = remaining / speed if speed > 0 else None\n    timing = f\'{format_duration(elapsed)}<{format_duration(eta)}, {speed:.2f} crop/s\' if speed > 0 else \'resume\'\n    print(f\'GPU {gpu_id}: {pct:5.1f}% {done}/{total} [{timing}, last={str(last_name)[:24]}]\', flush=True)\n\n\ndef infer_batch_with_fallback(model, processor, batch, device, gpu_id, cfg):\n    try:\n        return generate_batch(model, processor, batch, device, cfg[\'max_pixels_crop\'], cfg[\'max_new_tokens\'])\n    except torch.cuda.OutOfMemoryError:\n        print(f\'[GPU {gpu_id}] OOM on batch_size={len(batch)}; retrying one by one\', flush=True)\n        torch.cuda.empty_cache()\n        gc.collect()\n    except Exception as exc:\n        print(f\'[GPU {gpu_id}] batch failed: {exc}; retrying one by one\', flush=True)\n        torch.cuda.empty_cache()\n        gc.collect()\n\n    outputs = []\n    for sample in batch:\n        try:\n            outputs.extend(generate_batch(model, processor, [sample], device, cfg[\'max_pixels_crop\'], cfg[\'max_new_tokens\']))\n        except Exception as exc:\n            print(f\'[GPU {gpu_id}] failed sample {sample.get("source_file_name")}: {exc}\', flush=True)\n            outputs.append(\'\')\n            torch.cuda.empty_cache()\n            gc.collect()\n    return outputs\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--gpu-id\', type=int, required=True)\n    parser.add_argument(\'--samples-jsonl\', required=True)\n    parser.add_argument(\'--output-csv\', required=True)\n    parser.add_argument(\'--config-json\', required=True)\n    args = parser.parse_args()\n\n    cfg = json.loads(Path(args.config_json).read_text(encoding=\'utf-8\'))\n    samples = read_jsonl(args.samples_jsonl)\n    output_csv = Path(args.output_csv)\n    device = \'cuda:0\'\n    total = len(samples)\n\n    print(f\'[GPU {args.gpu_id}] visible CUDA devices={os.environ.get("CUDA_VISIBLE_DEVICES")} total_crops={total}\', flush=True)\n    print(f\'[GPU {args.gpu_id}] loading model\', flush=True)\n    model, processor = load_worker_model(cfg[\'model_id\'], cfg[\'lora_dir\'], device)\n    print(f\'[GPU {args.gpu_id}] model loaded\', flush=True)\n\n    done, results = load_partial(output_csv, cfg[\'resume_partials\'])\n    pending = [sample for sample in samples if sample[\'_sample_key\'] not in done]\n    print(f\'[GPU {args.gpu_id}] resume={len(done)} pending={len(pending)} partial={output_csv}\', flush=True)\n    print_progress(args.gpu_id, len(done), total, 0, 0.0, \'start\')\n\n    run_start = time.time()\n    run_done = 0\n    batch_size = int(cfg[\'batch_size\'])\n    for start in range(0, len(pending), batch_size):\n        batch = pending[start:start + batch_size]\n        outputs = infer_batch_with_fallback(model, processor, batch, device, args.gpu_id, cfg)\n        for sample, text in zip(batch, outputs):\n            results.append(result_row(sample, text))\n            done.add(sample[\'_sample_key\'])\n            run_done += 1\n\n        elapsed = max(1e-6, time.time() - run_start)\n        if run_done % int(cfg[\'progress_log_every\']) == 0 or len(done) == total:\n            print_progress(args.gpu_id, len(done), total, run_done, elapsed, batch[-1].get(\'source_file_name\'))\n\n        if len(results) % int(cfg[\'checkpoint_every\']) == 0:\n            save_partial(output_csv, results)\n            print(f\'[GPU {args.gpu_id}] checkpoint saved rows={len(results)} -> {output_csv}\', flush=True)\n\n    save_partial(output_csv, results)\n    print(f\'[GPU {args.gpu_id}] done {len(done)}/{total}; saved {output_csv}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
WORKER_SCRIPT.write_text(worker_source, encoding='utf-8')
WORKER_CONFIG.write_text(
    json.dumps(
        {
            'model_id': str(model_id),
            'lora_dir': str(lora_dir),
            'batch_size': int(BATCH_SIZE),
            'max_new_tokens': int(MAX_NEW_TOKENS),
            'max_pixels_crop': int(MAX_PIXELS_CROP),
            'checkpoint_every': int(CHECKPOINT_EVERY),
            'progress_log_every': int(PROGRESS_LOG_EVERY),
            'resume_partials': bool(RESUME_PARTIALS),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)
print('Worker script:', WORKER_SCRIPT)
print('Worker config:', WORKER_CONFIG)

checkpoint_dir = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else None
if checkpoint_dir and checkpoint_dir.exists():
    print('Restoring partial checkpoints from', checkpoint_dir)
    for src in checkpoint_dir.glob(f'{PARTIAL_PREFIX}*.csv'):
        dst = Path.cwd() / src.name
        shutil.copy2(src, dst)
        print('Copied', src, '->', dst)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Kaggle inference notebook.')

num_gpus = detect_num_gpus()
print('Using GPUs:', num_gpus)
chunks = [heldout_samples[i::num_gpus] for i in range(num_gpus)]
partials = []
processes = []

for gpu_id, chunk in enumerate(chunks):
    if not chunk:
        continue
    chunk_path = Path.cwd() / f'{PARTIAL_PREFIX}{gpu_id}_samples.jsonl'
    output_csv = f'{PARTIAL_PREFIX}{gpu_id}.csv'
    write_jsonl(chunk_path, chunk)
    partials.append(output_csv)

    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTHONUNBUFFERED'] = '1'
    cmd = [
        sys.executable,
        str(WORKER_SCRIPT),
        '--gpu-id',
        str(gpu_id),
        '--samples-jsonl',
        str(chunk_path),
        '--output-csv',
        output_csv,
        '--config-json',
        str(WORKER_CONFIG),
    ]
    print('Starting worker:', ' '.join(cmd), 'CUDA_VISIBLE_DEVICES=' + env['CUDA_VISIBLE_DEVICES'], flush=True)
    p = subprocess.Popen(cmd, cwd=str(Path.cwd()), env=env)
    processes.append((gpu_id, p))

for gpu_id, p in processes:
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'GPU worker {gpu_id} exited with code {rc}')

print('Partial outputs:', partials)


In [ ]:
def load_prediction_frames(paths):
    frames = []
    for path in paths:
        p = Path(path)
        if not p.exists():
            print('Missing partial:', p)
            continue
        frame = pd.read_csv(p).fillna('')
        frame = frame.drop_duplicates(subset=['sample_key'], keep='last')
        print('Partial', p, 'rows=', len(frame))
        frames.append(frame)
    if not frames:
        raise RuntimeError('No partial prediction CSVs found.')
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=['sample_key'], keep='last')


def make_submission_region(record, region, pred_by_key):
    bbox = canonical_bbox(region.get('bbox'))
    rtype = normalize_type(region.get('type'))
    text = ''
    if rtype in SCORABLE_TYPES:
        key = sample_key_from_parts(record.get('file_name'), bbox, rtype)
        text = pred_by_key.get(key, '')
        if not text and USE_METADATA_TEXT_FOR_UNPREDICTED:
            text = str(region.get('text') or '')
    return {'bbox': bbox, 'type': rtype, 'text': text}


def build_submission_rows(records, pred_by_key):
    rows = []
    for record in records:
        regions = [make_submission_region(record, region, pred_by_key) for region in (record.get('regions') or [])]
        image_name = Path(record.get('file_name') or '').name
        rows.append({'image': image_name, 'regions': json.dumps(regions, ensure_ascii=False)})
    return rows


pred_df = load_prediction_frames(partials)
ordered_keys = [sample['_sample_key'] for sample in heldout_samples]
pred_df['_order'] = pred_df['sample_key'].map({key: idx for idx, key in enumerate(ordered_keys)})
pred_df = pred_df.sort_values('_order').drop(columns=['_order'])
pred_df.to_csv(PREDICTION_CSV, index=False)

pred_by_key = dict(zip(pred_df['sample_key'].astype(str), pred_df['text'].astype(str)))
missing_manifest = [sample for sample in heldout_samples if sample['_sample_key'] not in pred_by_key]
print('Crop prediction CSV:', PREDICTION_CSV, 'rows=', len(pred_df))
print('Missing crop predictions:', len(missing_manifest))

submission_rows = build_submission_rows(metadata_records, pred_by_key)
submission_df = pd.DataFrame(submission_rows, columns=['image', 'regions'])
submission_df.to_csv(OUTPUT_CSV, index=False)
print('Wrote', OUTPUT_CSV, 'rows=', len(submission_df))
submission_df.head()


In [ ]:
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ['image', 'regions'], df.columns
assert len(df) == len(metadata_records), (len(df), len(metadata_records))

bad = []
region_counts = []
text_counts = []
for row in df.itertuples(index=False):
    try:
        regions = json.loads(row.regions)
        assert isinstance(regions, list)
        region_counts.append(len(regions))
        text_counts.append(sum(1 for item in regions if str(item.get('text') or '').strip()))
        for item in regions:
            assert set(item) == {'bbox', 'type', 'text'}
            assert isinstance(item['bbox'], list) and len(item['bbox']) == 4
            assert item['type'] in VALID_TYPES
    except Exception as exc:
        bad.append((row.image, str(exc)))
        if len(bad) >= 5:
            break

print('Bad rows:', bad[:5])
print('Images:', len(df))
print('Total regions:', sum(region_counts))
print('Regions with predicted text:', sum(text_counts))
print('Avg regions/page:', round(sum(region_counts) / max(1, len(region_counts)), 2))
print('Ready:', OUTPUT_CSV)
df.head()
